# 03 — Partitioning & Skew

**Objective:** Explore how Spark partitions data, why skew matters, and how reshuffling or salting can improve performance. This notebook is focused on interview-level explanations for senior/lead data engineering roles.

## How to map explain() to the Spark UI

Use `explain(mode="formatted")` together with the Spark UI to trace execution and debug performance: 
- Job tab: identify which action triggered the job, count the stages, and verify the job name or description.
- Stage tab: locate shuffle boundaries, identify the slowest stage, and find the stage that is the bottleneck.
- Task tab: inspect task runtime skew, disk spill, GC time, and input/output sizes. A single task taking ~10x longer is a strong skew signal.
- SQL tab: compare the physical plan with the `explain()` output, identify `Exchange` nodes, join strategy, filter pushdown, and any AQE decisions such as coalesced shuffle partitions.

This notebook uses partitioning and skew examples that are easy to map to the UI, so you can practice asking: "a job is taking 3x longer than expected; where do you start?"


In [21]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path().absolute().parent))

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('03-partitioning-skew')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
    .config('spark.python.worker.faulthandler.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} — UI: http://localhost:4040')
print('Default shuffle.partitions:', spark.conf.get('spark.sql.shuffle.partitions'))
print('Default parallelism:', spark.sparkContext.defaultParallelism)

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

## Why partitioning matters

Spark divides data into partitions. Each task processes one partition. In a distributed job, partition distribution determines parallelism, data locality, and CPU/memory balance.

Key interview points:
- `repartition()` creates a new set of partitions and can shuffle data
- `coalesce()` reduces partitions without a full shuffle when possible
- Skewed keys cause one or few partitions to contain most of the data
- Skew is the most common cause of stage bottlenecks in Spark

In [ ]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Skewed distribution: key 'A' owns 80% of all rows — simulates a hot-key scenario
# (e.g., one dominant customer_id, country_code, or event_type in a production table).
# With 1000 rows, the task assigned to key 'A' will process 8–20x more data than
# the tasks for B/C/D, making skew clearly visible in the Spark UI Task Metrics tab.
rows = (
    [(i, 'A', 'alpha') for i in range(1,   801)] +   # 800 rows — hot key (80%)
    [(i, 'B', 'beta')  for i in range(801,  901)] +   # 100 rows  (10%)
    [(i, 'C', 'gamma') for i in range(901,  961)] +   #  60 rows   (6%)
    [(i, 'D', 'delta') for i in range(961, 1001)]     #  40 rows   (4%)
)

schema = StructType([
    StructField('id',       IntegerType(), False),
    StructField('key',      StringType(),  True),
    StructField('category', StringType(),  True),
])

pdf = pd.DataFrame(rows, columns=['id', 'key', 'category'])
df = spark.createDataFrame(pdf, schema=schema)
df = df.repartition(8)  # round-robin across 8 partitions — source is balanced, skew appears after shuffle

In [5]:
df.explain(mode='extended')

== Parsed Logical Plan ==
Repartition 8, true
+- LocalRelation [id#0, key#1, category#2]

== Analyzed Logical Plan ==
id: int, key: string, category: string
Repartition 8, true
+- LocalRelation [id#0, key#1, category#2]

== Optimized Logical Plan ==
Repartition 8, true
+- LocalRelation [id#0, key#1, category#2]

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=58]
   +- LocalTableScan [id#0, key#1, category#2]



In [6]:
print(f'Total rows : {df.count()}')
print(f'Partitions : {df.rdd.getNumPartitions()}')

Total rows : 1000
Partitions : 8


![alt text](resources/repartition+count_two_phase_aggregation.svg)

In [7]:
df.groupBy('key').count().orderBy('key').show()

+---+-----+
|key|count|
+---+-----+
|  A|  800|
|  B|  100|
|  C|   60|
|  D|   40|
+---+-----+



In [ ]:
def partition_summary(df, label):
    counts = (
        df.groupBy(F.spark_partition_id().alias('pid'))
          .agg(F.count('*').alias('n'))
          .orderBy('pid')
          .collect()
    )
    print(f'=== {label} partition counts ===')
    for row in counts:
        print(f'Partition {row.pid}: {row.n} rows')
    print()

partition_summary(df, 'Initial DF')

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 20.0 failed 1 times, most recent failure: Lost task 5.0 in stage 20.0 (TID 64) (192.168.1.162 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:685)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1014)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:596)
	at org.apache.spark.InterruptibleIterator.hasNex

In [ ]:
# Source partitions are BALANCED because repartition(8) uses round-robin assignment.
# Skew only surfaces after a key-based shuffle (groupBy, join).
# This is the baseline to compare against the post-shuffle partition_summary below.
partition_summary(df, 'Source partitions — balanced (round-robin, not key-based)')
print()
print('Key distribution in the source data:')
df.groupBy('key').count().orderBy('key').show()
print('Observation: source partitions look balanced, but key A owns 80% of rows.')
print('After a groupBy(key), all 800 A-rows land in ONE shuffle partition — that is skew.')


## Inspecting skew in a shuffle

We will use a skewed key distribution to show how a `groupBy` can produce one heavy task. In this example, key `A` is highly skewed.

In [ ]:
grouped = df.groupBy('key').count()
print('GroupBy count partitions:', grouped.rdd.getNumPartitions())
grouped.explain(mode='formatted')
grouped.show(truncate=False)
partition_summary(grouped, 'GroupBy result')

### Spark UI — reading skew in the Task Metrics tab

After running the `groupBy` and `show()` above, open the UI at http://localhost:4040 and trace the execution:

**SQL tab** — find the query for `grouped.show()`:
- Read the plan bottom to top: `LocalTableScan` → `HashAggregate(partial)` → `Exchange hashpartitioning(key#..., 8)` → `AQEShuffleRead` → `HashAggregate(final)`
- The `Exchange` node is the stage boundary — everything below it is Stage 0, everything above is Stage 1
- Pre-execution shows `AdaptiveSparkPlan isFinalPlan=false`; after the action it shows `isFinalPlan=true` with `AQEShuffleRead`

**Stages tab — what you will actually see:**
- Stage 0: scan + repartition shuffle write — 8 tasks, Shuffle Write ≈ **125 rows each** (balanced, round-robin)
- Stage 1: read balanced partitions + `HashAggregate(Partial)` + groupBy shuffle write
  - Shuffle Read ≈ **125 rows per task** — still balanced, because repartition was round-robin
  - Shuffle Write ≈ **4 rows per task** — partial aggregation reduces each partition to one count per distinct key (A, B, C, D)
- Stage 2: `HashAggregate(Final)` — AQE coalesces to 4 tasks, one per non-empty key bucket; each reads **8 rows** (one partial count from each Stage 1 task)

> ⚠️ **Why skew is NOT visible in Task Metrics for `count()`:**
> The partial aggregation in Stage 1 reduces 125 rows → 4 rows per task *before* the shuffle.
> Stage 2 then processes exactly 8 rows per task regardless of key distribution.
> All task durations look equal — the skew in the source data is absorbed before anything crosses the network.

**Where the skew IS visible — `repartition(4, 'key')` + `partition_summary`:**
Hash-partitioning by key sends all rows with the same key to the same partition.
`partition_summary` will show the imbalance directly:
```
Partition X:  800 rows   ← all A-rows (hash('A') % 4)
Partition Y:  100 rows
Partition Z:   60 rows
Partition W:   40 rows
```
In the Spark UI this manifests as Task Metrics skew for operations that **cannot pre-reduce**, such as **joins**: the task for partition X processes 800 rows while others process 40–100, producing a clear duration outlier. That pattern — one task far to the right on the Duration histogram — is the classic skew signal. See Section 5 (joins) for an example.

**What to say in an interview:**
> "Skew in `groupBy().count()` is absorbed by partial aggregation before the shuffle, so you won't see it in Task Metrics. To detect skew, I'd look at `repartition(n, key)` + partition size inspection, or examine Task Metrics on a join or window function stage where all rows with the same key must be co-located without pre-reduction."

In [ ]:
# Post-execution plan — AQE decisions are now locked in (isFinalPlan=true).
# Compare with the pre-execution plan printed above:
#   Before: AdaptiveSparkPlan isFinalPlan=false | Exchange hashpartitioning(key, 8)
#   After:  AdaptiveSparkPlan isFinalPlan=true  | AQEShuffleRead (actual partition count)
# AQE coalesces the planned 8 shuffle partitions to however many are non-empty (≤4 keys).
print('Post-execution plan (AQE decisions finalized):')
grouped.explain(mode='formatted')


## Visible Skew: SortMergeJoin on a Hot Key

`groupBy().count()` hides skew because partial aggregation reduces each partition to one row per key *before* the shuffle — Stage 2 sees 8 tiny rows regardless of how imbalanced the source is.

**Joins are different.** In a `SortMergeJoin`, every row with key `A` must land in the same task on both sides so they can be matched. No pre-reduction is possible. All 80,000 A-rows hit one task while B/C/D tasks process 4,000–10,000 rows each. That imbalance shows up directly in Task Metrics Duration and Input Records columns.

This section shows the skew raw, then two fixes:
1. **AQE Skew Join** — Spark splits the hot partition automatically at runtime
2. **Broadcast Join** — eliminate the shuffle on the large side entirely (best fix when the right side is small)

In [5]:
import time

# 100,000 rows with 80/10/6/4 key distribution — spark.range is fully JVM-native.
large_skewed = (
    spark.range(0, 100_000_0)
    .withColumn('key',
        F.when(F.col('id') < 80_000_0, F.lit('A'))
         .when(F.col('id') < 90_000_0, F.lit('B'))
         .when(F.col('id') < 96_000_0, F.lit('C'))
         .otherwise(F.lit('D'))
    )
    .withColumn('value', (F.rand(seed=42) * 100).cast('double'))
    .repartition(8)  # round-robin: source partitions are balanced; skew only appears after key shuffle
)

# SQL VALUES clause creates the lookup entirely in the JVM — no Python workers.
# spark.createDataFrame(python_list, column_names) without an explicit schema backs the
# DataFrame with a Python-serialized source that spawns a Python worker each time it is
# read; on Spark 4.x that worker crashes. VALUES avoids the Python path entirely.
lookup = spark.sql("""
    SELECT key, CAST(multiplier AS DOUBLE) AS multiplier FROM VALUES
        ('A', 1.5), ('B', 2.0), ('C', 3.0), ('D', 4.0)
    AS t(key, multiplier)
""")

print('Key distribution in large_skewed — skew is in the DATA, not yet in partitions:')
large_skewed.groupBy('key').count().orderBy('key').show()
print('After a key-based shuffle (join or repartition by key), key A owns 80% of one partition.')

Key distribution in large_skewed — skew is in the DATA, not yet in partitions:
+---+------+
|key| count|
+---+------+
|  A|800000|
|  B|100000|
|  C| 60000|
|  D| 40000|
+---+------+

After a key-based shuffle (join or repartition by key), key A owns 80% of one partition.


In [6]:
# ── APPROACH 1: Skewed SortMergeJoin — raw skew, no fix ─────────────────────
# broadcast disabled → Spark must shuffle both sides by key → SortMergeJoin
# skewJoin disabled  → AQE does NOT split the hot A-partition
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'false')

skewed_join = (
    large_skewed
    .join(lookup, on='key')
    .select('id', 'key', (F.col('value') * F.col('multiplier')).alias('weighted'))
)
print('── Skewed SortMergeJoin plan (two Exchange nodes, one per side) ──')
skewed_join.explain(mode='formatted')

t0 = time.perf_counter()
skewed_join.count()
print(f'Skewed SMJ: {time.perf_counter()-t0:.3f}s')
print()
print('→ Spark UI  /  Stages tab  /  SortMergeJoin stage  /  Task Metrics')
print('  Input Records: one task ≈ 800,001 rows | others ≈ 40,001–100,001 rows')
print('  Duration histogram: one bar far to the right — key A holds the stage hostage.')


── Skewed SortMergeJoin plan (two Exchange nodes, one per side) ──
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Exchange (3)
      :        +- Project (2)
      :           +- Range (1)
      +- Sort (8)
         +- Exchange (7)
            +- LocalTableScan (6)


(1) Range
Output [1]: [id#36L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(2) Project
Output [3]: [id#36L, CASE WHEN (id#36L < 800000) THEN A WHEN (id#36L < 900000) THEN B WHEN (id#36L < 960000) THEN C ELSE D END AS key#37, (rand(42) * 100.0) AS value#38]
Input [1]: [id#36L]

(3) Exchange
Input [3]: [id#36L, key#37, value#38]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=671]

(4) Exchange
Input [3]: [id#36L, key#37, value#38]
Arguments: hashpartitioning(key#37, 8), ENSURE_REQUIREMENTS, [plan_id=677]

(5) Sort
Input [3]: [id#36L, key#37, value#38]
Arguments: [key#37 ASC NULLS FIRST], false, 0


In [7]:
# ── APPROACH 1: Skewed SortMergeJoin — raw skew, no fix ─────────────────────
# Three AQE features must be individually disabled to expose the raw per-key skew:
#
#   autoBroadcastJoinThreshold = -1
#     Forces SortMergeJoin — both sides shuffle by key.
#
#   coalescePartitions.enabled = false
#     Our 100K rows (~3 MB) are far below AQE's 64 MB advisory partition size.
#     With coalescing ON, AQE merges all non-empty key partitions into 1 task
#     *before* the Sort/Join — hiding the A/B/C/D imbalance behind "1 task handles all".
#     Disabling it keeps the planned 8 partitions intact so each key lands in its own task.
#
#   skewJoin.enabled = false
#     Prevents AQE from automatically splitting the large A partition at runtime.
#     We want to see the raw outlier before showing the fix.
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'false')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'false')

skewed_join = (
    large_skewed
    .join(lookup, on='key')
    .select('id', 'key', (F.col('value') * F.col('multiplier')).alias('weighted'))
)
print('── Skewed SortMergeJoin plan (two Exchange nodes, one per side) ──')
skewed_join.explain(mode='formatted')

t0 = time.perf_counter()
skewed_join.count()
print(f'Skewed SMJ: {time.perf_counter()-t0:.3f}s')
print()
print('→ Spark UI / Stages tab / SortMergeJoin stage / Task Metrics')
print('  8 tasks total (one per planned partition); 4 non-empty, 4 trivially empty.')
print('  Input Records: task A ≈ 80,001 | task B ≈ 10,001 | C ≈ 6,001 | D ≈ 4,001')
print('  Duration histogram: one bar far to the right — key A holds the stage hostage.')

── Skewed SortMergeJoin plan (two Exchange nodes, one per side) ──
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Exchange (3)
      :        +- Project (2)
      :           +- Range (1)
      +- Sort (8)
         +- Exchange (7)
            +- LocalTableScan (6)


(1) Range
Output [1]: [id#36L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(2) Project
Output [3]: [id#36L, CASE WHEN (id#36L < 800000) THEN A WHEN (id#36L < 900000) THEN B WHEN (id#36L < 960000) THEN C ELSE D END AS key#37, (rand(42) * 100.0) AS value#38]
Input [1]: [id#36L]

(3) Exchange
Input [3]: [id#36L, key#37, value#38]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=913]

(4) Exchange
Input [3]: [id#36L, key#37, value#38]
Arguments: hashpartitioning(key#37, 8), ENSURE_REQUIREMENTS, [plan_id=919]

(5) Sort
Input [3]: [id#36L, key#37, value#38]
Arguments: [key#37 ASC NULLS FIRST], false, 0


In [8]:
# ── APPROACH 2: AQE Skew Join — Spark splits the hot partition at runtime ────
# Re-enable coalescePartitions (it helps on large datasets, and skewJoin works alongside it).
# AQE detects that partition A is >> median size and automatically splits it.
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '3')
# Our A-partition is ~3 MB; lower the byte threshold so AQE triggers (default = 256 MB)
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes', '1048576')  # 1 MB

aqe_join = (
    large_skewed
    .join(lookup, on='key')
    .select('id', 'key', (F.col('value') * F.col('multiplier')).alias('weighted'))
)
t0 = time.perf_counter()
aqe_join.count()
print(f'\nAQE skew join: {time.perf_counter()-t0:.3f}s')
print('Post-execution plan — look for SkewJoin=true on the SortMergeJoin node:')
aqe_join.explain(mode='formatted')
print('→ Spark UI: A-partition split into sub-tasks; Duration histogram is roughly uniform.')


AQE skew join: 0.767s
Post-execution plan — look for SkewJoin=true on the SortMergeJoin node:
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Exchange (3)
      :        +- Project (2)
      :           +- Range (1)
      +- Sort (8)
         +- Exchange (7)
            +- LocalTableScan (6)


(1) Range
Output [1]: [id#36L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(2) Project
Output [3]: [id#36L, CASE WHEN (id#36L < 800000) THEN A WHEN (id#36L < 900000) THEN B WHEN (id#36L < 960000) THEN C ELSE D END AS key#37, (rand(42) * 100.0) AS value#38]
Input [1]: [id#36L]

(3) Exchange
Input [3]: [id#36L, key#37, value#38]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=1376]

(4) Exchange
Input [3]: [id#36L, key#37, value#38]
Arguments: hashpartitioning(key#37, 8), ENSURE_REQUIREMENTS, [plan_id=1382]

(5) Sort
Input [3]: [id#36L, key#37, value#38]
Arguments: [key#

In [9]:
# ── APPROACH 3: Broadcast Hash Join — eliminate the shuffle on the large side ─
# The lookup is 4 rows — send it to every executor in a BroadcastExchange.
# large_skewed never shuffles: each task joins its local round-robin partition.
# Skew in large_skewed does not matter — partitions are balanced from repartition(8).
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'false')  # broadcast alone is sufficient

broadcast_join = (
    large_skewed
    .join(F.broadcast(lookup), on='key')
    .select('id', 'key', (F.col('value') * F.col('multiplier')).alias('weighted'))
)
print('\n── Broadcast join plan (no Exchange on large_skewed side) ──')
broadcast_join.explain(mode='formatted')

t0 = time.perf_counter()
broadcast_join.count()
print(f'Broadcast join: {time.perf_counter()-t0:.3f}s')
print('→ Spark UI: NO SortMergeJoin stage. Tasks process ~12,500 rows each — round-robin balance.')

# Reset all configs to session defaults
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes', str(256 * 1024 * 1024))
spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '5')
print('\nAll configs reset to defaults.')


── Broadcast join plan (no Exchange on large_skewed side) ──
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin Inner BuildRight (6)
      :- Exchange (3)
      :  +- Project (2)
      :     +- Range (1)
      +- BroadcastExchange (5)
         +- LocalTableScan (4)


(1) Range
Output [1]: [id#36L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(2) Project
Output [3]: [id#36L, CASE WHEN (id#36L < 800000) THEN A WHEN (id#36L < 900000) THEN B WHEN (id#36L < 960000) THEN C ELSE D END AS key#37, (rand(42) * 100.0) AS value#38]
Input [1]: [id#36L]

(3) Exchange
Input [3]: [id#36L, key#37, value#38]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=1407]

(4) LocalTableScan
Output [2]: [key#39, multiplier#41]
Arguments: [key#39, multiplier#41]

(5) BroadcastExchange
Input [2]: [key#39, multiplier#41]
Arguments: HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=1412]

(6) BroadcastHashJoin
Left keys [1]: [key#37]


### Spark UI — reading the three approaches side by side

Open http://localhost:4040 and find the three `count()` jobs in order.

---

## Approach 1 — Skewed SortMergeJoin: detailed UI analysis

**SQL tab — what to find:**
- Two `Exchange hashpartitioning(key, 8)` nodes — one per join side
- `SortMergeJoin [key]` between them — all same-key rows must meet in the same task
- AQE is disabled for coalescing, so the 8 planned partitions survive into the join stage

**SortMergeJoin stage — Shuffle Read task metrics (what you will actually see):**

| Task | Shuffle Read Size | Shuffle Read Records | Notes |
|---|---|---|---|
| Key A | **12.4 KiB** | **80,001** | 80,000 from `large_skewed` + 1 from `lookup` |
| Key B + C | 3 KiB | 16,002 | 10,000 B-rows + 6,000 C-rows + 2 lookup rows — **hash collision** |
| Key D | 1.1 KiB | 4,001 | 4,000 + 1 |
| 5 tasks | 59 B | 1 | empty — no matching rows (partial count = 0) |

**What each observation tells you:**

**Hash collision (B+C in the same task):** `hash('B') % 8 == hash('C') % 8` in this run. Both keys land in the same shuffle bucket. This is expected behaviour — with 8 shuffle partitions and 4 keys, collisions are possible and visible in Task Metrics as a task with mixed keys. It is not an error; it means one task processes the combined load of two keys. In production, if a "hot bucket" keeps recurring, you can increase `spark.sql.shuffle.partitions` to spread keys across more buckets and reduce collision probability.

**Bytes-per-record compression ratio:** The three non-empty tasks show different bytes-per-record ratios:
- Key A: 12.4 KiB / 80,001 records ≈ **0.16 bytes/record**
- B+C bucket: 3 KiB / 16,002 records ≈ 0.19 bytes/record
- Key D: 1.1 KiB / 4,001 records ≈ **0.28 bytes/record**

Larger, homogeneous partitions compress better — the shuffle file codec amortizes its overhead across more records, and long runs of the same key value compress efficiently. Smaller partitions pay more overhead per record. This is why the bytes-per-record ratio decreases as partition size increases. In practice this means a 20× records imbalance translates to less than 20× bytes imbalance — but Records is still the reliable skew signal, not Bytes.

**The 59 B / 1 on all 8 tasks (Shuffle Write):** Every task writes a result regardless of whether it processed any input rows. An empty task writes a single partial aggregate result of 0 records — that is the `HashAggregate(partial)` writing its empty count. The 59 bytes is the serialization overhead for one row containing a zero count. This is normal; it means the scheduler assigned all 8 partitions (as planned) even though only 3 had actual data. The 5 empty tasks are fast (sub-millisecond) but still consume task-scheduling overhead on the driver.

**Duration is unreliable in local mode — use Records:** In local mode all tasks run on the same JVM thread pool with shared CPU and memory. The 80,001-record task may finish in 100ms while the 4,001-record task reports 62ms — a 1.6× duration ratio for a 20× records ratio. This is because local-mode task overhead (serialization, driver round-trips) dominates over actual compute time on small datasets. In a real distributed cluster with 100M-row tables, the 20× records imbalance translates directly to ~20× wall-clock time on that executor — meaning the A-task would stall the stage for 20× longer than the D-task. **Always use Input Records as the skew signal; use Duration only to confirm on large production jobs.**

**The stage completion rule:** A Spark stage cannot close until its slowest task finishes. The 80,001-record A-task is the stage wall — all other executors sit idle once their tasks complete, waiting for A. On a 32-executor cluster processing 800M rows for key A, that idle time is 31 executors × (A's runtime − median runtime) = direct cluster waste. This is exactly what you see in the Duration histogram as the one bar far to the right.

**Production implication:** A 20× records imbalance in a join stage means the stage takes 20× as long as it would with a perfectly balanced key. If the balanced version took 3 minutes per stage, the skewed version takes ~60 minutes — and every downstream stage waits.

---

**Approach 1 — Decision trigger:**

```
SortMergeJoin stage Task Metrics shows one outlier task?
  → Check Input Records: is one task >> 5× the median?
      → That is your hot key. Identify which key with a groupBy().count().orderBy(count.desc()).
      → Is the other join side small (< 10 MB)? → Approach 3 (broadcast) is the simplest fix.
      → Is the other join side large? → Approach 2 (AQE skew join) or manual salting.
```

---

**Approach 2 — AQE Skew Join (SQL tab, post-execution plan):**
- Look for `SkewJoin=true` on the `SortMergeJoin` node
- `AQEShuffleRead` wraps the Exchange with `isSkewedPartition` markers
- AQE split the 80,000-row A partition into sub-partitions (~20,000 each) and ran one task per sub-partition
- The lookup's A partition (1 row) is read once per sub-task (replicated internally)

**Approach 2 — Task Metrics:** Duration histogram is roughly uniform — no single outlier.

---

**Approach 3 — Broadcast Hash Join (SQL tab):**
- `BroadcastExchange HashedRelationBroadcastMode` on the `lookup` branch only
- **No `Exchange` on `large_skewed`** — the large side is never shuffled (each task joins its local round-robin partition) as we broadcast the lookup table to each parition/executor if it fits in memory
- `BroadcastHashJoin [key]` — each executor joins its local partition against the in-memory lookup copy

**Approach 3 — Task Metrics:** All 8 tasks process ≈ 12,500 rows (the round-robin balanced partitions of `large_skewed`). No SortMergeJoin stage exists at all. The skew in the source data is irrelevant because the large side's partitions were balanced by the initial `repartition(8)` (round-robin, not key-based).

---

**Decision rule for production:**

```
SortMergeJoin stage has an outlier task?
  → Is the non-skewed side < autoBroadcastJoinThreshold (10 MB default)?
      YES → use F.broadcast(small_df)          — zero extra shuffles, simplest fix
      NO  → Is AQE skew join detecting it?      (check post-exec plan for SkewJoin=true)
              YES → tune skewedPartitionFactor and skewedPartitionThresholdInBytes
              NO  → manual salting               — replicate small side, salt large side
```

**Interview answer — reading Task Metrics for skew:**
> "I open the SQL tab and find the `SortMergeJoin` stage. In Task Metrics I look at Input Records per task — if one task shows 20× the median record count, that's a hot key. In our experiment, key A had 80,001 records versus key D's 4,001 — a 20× imbalance. I also noticed that keys B and C landed in the same shuffle bucket due to a hash collision with 8 partitions, so their records combined into one task. The 5 empty tasks each wrote 59 bytes — that's the partial aggregate writing a count of zero, normal but a signal of over-partitioning. For the fix: if the smaller join side fits in memory I use `F.broadcast()` — the large side never shuffles and the skew becomes irrelevant. If both sides are large I enable AQE skew join by lowering `skewedPartitionThresholdInBytes` to match the actual partition size. Duration histograms in local mode are misleading because task overhead dominates; I always use Records ratio as the primary skew signal."

## Repartitioning by key

### What "hash partitioning by key" means

`repartition(n)` without a key distributes rows **round-robin**: row 1 goes to partition 0, row 2 to partition 1, and so on, wrapping around. No row has a guaranteed landing spot — the result is balanced but unordered by key.

`repartition(n, 'key')` distributes rows by **`hash(key_value) % n`**. Every row where `key = 'A'` always lands in the same partition, deterministically. The partition that "owns" A will hold ALL 800 A-rows. This is what "hash partitioning by key" means: the key value determines the destination partition, not the row's position in the sequence.

### What it is actually for

Pre-partitioning by key is useful when the **same key column** drives multiple downstream operations. You pay the shuffle cost once upfront and reuse the alignment across all of them:

```
# Without pre-partitioning: each op shuffles independently
df.join(other, on='key')          # Exchange 1
  .groupBy('key').agg(...)        # Exchange 2
  .withColumn(...window on key)   # Exchange 3

# With pre-partitioning: pay once, all subsequent ops on the same key reuse it
df.repartition(4, 'key')         # Exchange 1 — all rows aligned by key
  .join(other_aligned, on='key') # no Exchange — already co-located
  .groupBy('key').agg(...)       # no Exchange — same partitioning reused
```

This matters in pipelines that chain joins, window functions, and aggregations on the same key. The single upfront shuffle eliminates all the downstream ones.

### What it does NOT do

**Pre-partitioning does not reduce skew** — it moves the skew from the shuffle into the partition. After `repartition(4, 'key')`, the partition assigned to `'A'` holds all 800 A-rows. A `groupBy('key')` on top still processes 800 rows in one task versus 40–100 in the others. The imbalance is identical; it just arrived via a different route.

**To fix skew, use salting** — the section after this one shows how.

In [ ]:
repartitioned = df.repartition(4, 'key')
print('Repartitioned DF partitions:', repartitioned.rdd.getNumPartitions())
partition_summary(repartitioned, 'Repartitioned DF by key')
repartitioned_grouped = repartitioned.groupBy('key').count()
repartitioned_grouped.explain(mode='formatted')
repartitioned_grouped.show(truncate=False)

### Spark UI — repartitioned by key

**SQL tab — what the plan shows:**

Run `repartitioned_grouped.explain(mode='formatted')` and count the `Exchange` nodes:

- Without the upfront `repartition(4, 'key')`: one `Exchange hashpartitioning(key, 8)` for the `groupBy`
- With `repartition(4, 'key')` then `groupBy('key')`: **two** Exchange nodes appear
  - Exchange 1: `RoundRobinPartitioning(8)` from `df.repartition(8)` (the original balanced source)
  - Exchange 2: `hashpartitioning(key, 4)` from the explicit `repartition(4, 'key')`
  - Exchange 3: `hashpartitioning(key, 8)` from the `groupBy` itself

The `groupBy` still triggers its own Exchange because it needs `spark.sql.shuffle.partitions=8` partitions and the pre-repartitioned data has 4. Spark does not detect that the existing key-based partitioning satisfies the `groupBy` requirement when partition counts differ. **Pre-partitioning only eliminates a downstream shuffle when the key AND the partition count both match.**

**Task Metrics — skew is unchanged:**

| Task | Input Records | Conclusion |
|---|---|---|
| Key A | 800 | All A-rows still in one partition |
| Key B | 100 | |
| Key C | 60 | |
| Key D | 40 | |

Task A is still the outlier in the Duration histogram. `repartition(n, 'key')` moved the skew into an explicit partition — it did not break it up.

**Takeaway:** Use `repartition(n, 'key')` when multiple *different* operations on the same key follow in sequence and you can match the partition count across all of them. It is a shuffle-elimination technique, not a skew-fix. Use salting (next section) to actually distribute a hot key across multiple tasks.

## `repartition()` vs `coalesce()`: Exchange or no Exchange

Both change the number of partitions. The plan tells you which one was used.

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffle | Full shuffle — `Exchange` node in the plan | Narrow merge — no `Exchange` |
| Direction | Increase or decrease | Decrease only |
| Balance | Round-robin (or hash if key given) — equal partition sizes | Best-effort — upstream skew is preserved |
| Use case | Key-based partitioning for joins/windows; increasing parallelism | Reduce partition count before a write without the extra shuffle cost |
| Plan node | `Exchange RoundRobinPartitioning(n)` | `Coalesce n` — no Exchange above it |

**Interview answer:**
> "`coalesce()` avoids the extra `Exchange` node, so it is the right choice when you are *reducing* partitions before writing — for example, collapsing 200 shuffle partitions to 10 output files. `repartition()` is correct when you need all partitions to be equal in size or need to hash-partition by a key for a subsequent join or window function."

In [ ]:
# repartition() vs coalesce() — observe the Exchange node difference in the plan.
# repartition(2) introduces a full shuffle (Exchange node); coalesce(2) does not.

print('── repartition(2): full shuffle — look for Exchange in the plan ──')
df.repartition(2).explain(mode='formatted')
# Expect: Exchange RoundRobinPartitioning(2)
# All rows are re-distributed across 2 partitions via network shuffle.

print('\n── coalesce(2): narrow merge — no Exchange node ──')
df.coalesce(2).explain(mode='formatted')
# Expect: Coalesce 2 directly above the scan — no Exchange, no network transfer.

# Verify partition distribution for each
print('\n── Partition row counts ──')
partition_summary(df.repartition(2), 'repartition(2) — round-robin, balanced')
partition_summary(df.coalesce(2),    'coalesce(2)    — merged without shuffle, may be uneven')

## Skew mitigation with salting

When one key dominates the data, add a salt to the key before shuffling, then remove the salt after the aggregation. This spreads skewed key values across multiple partitions.

In [ ]:
salted = df.withColumn('salted_key', F.concat(F.col('key'), F.lit('_'), (F.rand() * 3).cast('int')))
salted = salted.repartition(8, 'salted_key')
print('Salted DF partitions:', salted.rdd.getNumPartitions())
partition_summary(salted, 'Salted DF')
salted_grouped = (
    salted.groupBy('salted_key', 'key')
    .count()
)
salted_grouped.show(truncate=False)

final_grouped = (
    salted_grouped
    .groupBy('key')
    .agg(F.sum('count').alias('count'))
)
final_grouped.show(truncate=False)

### Spark UI — salting effect on task distribution

**Stages tab — compare stage count with the unskewed baseline:**
- Without salting: 2 stages (scan+partial → shuffle+final)
- With salting: **3 stages** — one extra shuffle for the two-pass aggregation
  - Stage 0: scan → partial agg by `salted_key` (A_0/A_1/A_2/B_0/...) → Exchange
  - Stage 1: shuffle read → final agg by `salted_key` → Exchange
  - Stage 2: shuffle read → re-aggregate by original `key`

**Task Metrics — Stage 1 (the critical first shuffle-read stage):**

| | Skewed (no salt) | Salted |
|---|---|---|
| Task for 'A' | 800 rows, outlier | Gone — split into A_0 (~267), A_1 (~267), A_2 (~266) |
| Max task records | 800 | ~267 (3× reduction in hot-key size) |
| Duration histogram | One outlier bar | Roughly uniform across 7 tasks (4 keys × salt 0–2, minus empties) |

**Trade-off visible in the UI:**
- Salting adds one `Exchange` node in the SQL tab (extra shuffle write + read)
- The extra stage is the cost; the more-uniform Task Metrics histogram is the benefit
- For small local data the timing difference is tiny; on 100M-row production data, the 'A' task that stalled the stage for 30 minutes now completes in < 5 minutes

**SQL tab — count the Exchange nodes:**
- Without salting: 1 Exchange (one groupBy → one shuffle)
- With salting: 2 Exchange nodes (two groupBys → two shuffles)
- Rule of thumb: each Exchange = one stage boundary = one network round-trip for all data

**When to use salting in a real job:**
If the Task Metrics Duration histogram shows one task > 5× the median and that task has many more Input Records than the others, salting is the correct mitigation. If all tasks are slow but roughly equal, the problem is data volume, not skew — tune parallelism instead.


## Key takeaway

- Data partitioning controls task parallelism and workload distribution.
- `repartition()` shuffles data; use it when you need explicit partitioning by key.
- `coalesce()` is best for reducing partitions without a full shuffle.
- Skewed keys lead to overloaded partitions and slow stages.
- Salting is a practical skew mitigation technique in interview discussions.

## Applied UI Analysis: Sections 5–8

The sections below extend the partitioning and skew concepts into four adjacent areas — joins, filter pushdown, AQE coalescing, and production debugging. Each section shows what the plan looks like and how to read it in the Spark UI.

These are **previews** — enough to recognize the pattern and answer the interview question. Full mechanism coverage is in dedicated notebooks:
- **NB04** — Joins (all four strategies, hints, bucketing, skew join handling)
- **NB05** — Catalyst optimizer (predicate pushdown rules, four plan stages, CBO, Tungsten)
- **NB06** — AQE deep dive (coalescing tuning, SMJ→BHJ conversion, Dynamic Partition Pruning)

Read the plans here to build intuition. Return to the dedicated notebooks for the mechanism behind each decision.

## 5 — Joins: Broadcast vs Sort-Merge
*Full join coverage (all four strategies, hints, bucketing, skew join handling) → NB04.*

The **join strategy** is the most impactful execution decision visible in the SQL tab and the most common cause of silent performance regressions. From a partitioning perspective, the key question is: how many shuffles does the join introduce?

| Strategy | When Spark chooses it | Plan nodes | Shuffle cost |
|---|---|---|---|
| `BroadcastHashJoin` | Small table < `autoBroadcastJoinThreshold` (10 MB default) | `BroadcastExchange` on small side | 0 shuffles on small side |
| `SortMergeJoin` | Both tables exceed threshold | `Exchange` + `Sort` on **both** sides | 2 shuffles |

**The regression pattern:** A pipeline that joined a 500K-row fact table with a 10K-row lookup ran in 3 minutes. After a data reload the lookup grew to 15M rows. Spark silently switched from BroadcastHashJoin → SortMergeJoin. Runtime jumped to 25 minutes. No code changed. The SQL tab catches this in 30 seconds.

**How to force the join you want:**
- `F.broadcast(small_df)` — explicit hint, always broadcasts regardless of size
- `spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '50m')` — raise the auto-threshold


In [ ]:
import time, random
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Small lookup table (4 rows — will be broadcast when threshold allows)
products_data = [
    (1, 'widget',    'hardware',  9.99),
    (2, 'gadget',    'hardware', 24.99),
    (3, 'service_a', 'software', 99.99),
    (4, 'service_b', 'software', 49.99),
]
products_schema = StructType([
    StructField('product_id',   IntegerType(), False),
    StructField('product_name', StringType(),  True),
    StructField('category',     StringType(),  True),
    StructField('unit_price',   DoubleType(),  True),
])
products_df = spark.createDataFrame(products_data, schema=products_schema)

# Large fact table — 1000 transactions
random.seed(42)
txn_data = [(i, random.randint(1, 4), random.randint(1, 10), float(random.randint(10, 200)))
            for i in range(1, 1001)]
txn_schema = StructType([
    StructField('txn_id',     IntegerType(), False),
    StructField('product_id', IntegerType(), True),
    StructField('quantity',   IntegerType(), True),
    StructField('amount',     DoubleType(),  True),
])
txn_df = spark.createDataFrame(txn_data, schema=txn_schema)
print(f'products_df: {products_df.count()} rows  |  txn_df: {txn_df.count()} rows')

# ── Sort-Merge Join: disable auto-broadcast to force SMJ ─────────────────────
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

smj = (txn_df.join(products_df, 'product_id')
             .groupBy('category').agg(F.sum('amount').alias('total')))

print('\n── Sort-Merge Join (broadcast disabled — 2 Exchange nodes, one per side) ──')
smj.explain(mode='formatted')
# Look for: SortMergeJoin, Exchange on BOTH the txn and products branches, Sort on both sides

# ── Broadcast Hash Join: re-enable and use explicit hint ─────────────────────
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))

bhj = (txn_df.join(F.broadcast(products_df), 'product_id')
             .groupBy('category').agg(F.sum('amount').alias('total')))

print('\n── Broadcast Hash Join (explicit hint — BroadcastExchange on small side only) ──')
bhj.explain(mode='formatted')
# Look for: BroadcastHashJoin, BroadcastExchange on products side, NO Exchange on txn side

print('\n── Run both actions and compare job durations in the UI ──')
t0 = time.perf_counter(); smj.show(); print(f'SMJ: {time.perf_counter()-t0:.3f}s')
t0 = time.perf_counter(); bhj.show(); print(f'BHJ: {time.perf_counter()-t0:.3f}s')


### Reading join plans in the SQL tab

Open the SQL tab and find the two queries for `smj.show()` and `bhj.show()`.

**Sort-Merge Join — nodes to find:**
- `SortMergeJoin [product_id#...]` — the join operator
- `Exchange hashpartitioning(product_id#..., 8)` appears on **both** the left (txn) and right (products) branches
- `Sort [product_id#... ASC]` on both branches — rows must be sorted before the merge can begin
- Two `Exchange` nodes = two shuffles = at least two stage boundaries before the join output

**Broadcast Hash Join — nodes to find:**
- `BroadcastHashJoin [product_id#...]` — the join operator
- `BroadcastExchange HashedRelationBroadcastMode(...)` on the **products branch only** — entire small table is sent to all executors
- **No `Exchange` on the txn_df branch** — the large table is never shuffled for the join
- One fewer stage boundary compared to SMJ

**Stages tab comparison:**

| | Sort-Merge Join | Broadcast Hash Join |
|---|---|---|
| Stage for txn shuffle | Yes (`Exchange` on txn) | No |
| Stage for products shuffle | Yes (`Exchange` on products) | No (broadcast instead) |
| Join + aggregate stage | Yes | Yes |
| Total stages (approximate) | 3 | 1–2 |

**Regression detection:**  
If a query's SQL tab used to show `BroadcastHashJoin` and now shows `SortMergeJoin`, the small table grew past `spark.sql.autoBroadcastJoinThreshold` (default 10 MB). Solutions: use an explicit `F.broadcast()` hint; raise the threshold; or pre-filter the large table before the join to keep the small table "small" relative to the threshold.


## 6 — Filter Pushdown: Where the Filter Lives in the Plan
*Full Catalyst coverage (all optimization rules, the four plan stages, CBO, Tungsten) → NB05.*

The **position of a `Filter` node relative to an `Exchange` node** in the SQL tab tells you whether Spark is shuffling data that will be discarded. This is a partitioning cost question — it determines how much data crosses the network.

| Filter position | SQL tab node order | Effect |
|---|---|---|
| Below Exchange (early) | `Scan → Filter → Aggregate → Exchange` | Only matching rows are shuffled |
| Above Exchange (late) | `Scan → Aggregate → Exchange → Filter` | All rows are shuffled, then filtered |

Catalyst's predicate pushdown rule automatically moves filters as close to the data source as possible. But filters on **aggregated columns** (e.g., `HAVING total > 500`) cannot be pushed below the aggregation — they must run after the `groupBy` completes.


In [ ]:
# ── Early filter: Catalyst predicate pushdown places it before the shuffle ───
# txn_df must be defined (run Section 5 first).
early_filter = (
    txn_df
    .filter(F.col('amount') > 50.0)      # filter on raw column — pushed below the Exchange
    .groupBy('product_id')
    .agg(F.sum('amount').alias('total'))
)
print('── Early filter (Filter node appears BELOW Exchange in the plan) ──')
early_filter.explain(mode='formatted')
# The Filter (amount > 50.0) sits between LocalTableScan and HashAggregate(partial).
# Only filtered rows are written to the shuffle files.

# ── Late filter: applied to aggregated result — stays above the aggregation ──
late_filter = (
    txn_df
    .groupBy('product_id')
    .agg(F.sum('amount').alias('total'))
    .filter(F.col('total') > 500.0)      # filter on aggregated column — cannot move below agg
)
print('\n── Late filter (Filter node appears ABOVE HashAggregate/Exchange) ──')
late_filter.explain(mode='formatted')
# The Filter (total > 500.0) appears above HashAggregate(final) — after all shuffling is done.
# The mechanism behind why Catalyst moves or keeps filters is covered in NB05.


### Filter position in the SQL tab — reading bottom to top

The SQL tab physical plan reads from **bottom (data source) to top (result)**. The `Exchange` node is the stage boundary. The position of `Filter` relative to `Exchange` tells you how much data crosses the network.

**Early filter — Filter BELOW Exchange (pushed down toward the scan):**

```
LocalTableScan (txn_df)
  └── Filter (amount > 50.0)          ← runs before the shuffle
        └── HashAggregate (partial)
              └── Exchange             ← only rows passing the filter are shuffled
                    └── HashAggregate (final)
```

Only rows with `amount > 50` cross the Exchange. The shuffle moves less data.

**Late filter — Filter ABOVE Exchange (applied after aggregation):**

```
LocalTableScan (txn_df)
  └── HashAggregate (partial)
        └── Exchange                   ← ALL rows are shuffled first
              └── HashAggregate (final)
                    └── Filter (total > 500.0)   ← runs after the shuffle
```

All rows go through the Exchange before the filter discards them.

**Interview answer:**
> "I'd look at the SQL tab and find where the Filter node sits relative to the Exchange. If it's above the Exchange, that filter runs after the shuffle — it's reading and moving data that gets thrown away. The fix is to apply the filter before the `groupBy`, which Catalyst will automatically push past the scan."

*How Catalyst decides to move filters — the predicate pushdown rule, the four plan stages, and cost-based optimization — is covered in NB05.*


## 7 — AQE: Coalesced Shuffle Partitions
*Full AQE coverage (all three features, Dynamic Partition Pruning, SMJ→BHJ conversion, skew join handling) → NB06.*

`spark.sql.shuffle.partitions` sets the **planned** partition count. After each shuffle stage writes its output files, AQE reads the actual file sizes and sets the **real** partition count for the next stage. The most visible result: 200 planned partitions often coalesce to a handful of non-empty ones.

**What to observe in the plan:**
- Pre-execution: `Exchange hashpartitioning(key#..., 200)` — Spark planned 200 partitions
- Post-execution: `AQEShuffleRead` node replaces/wraps the Exchange — shows actual partition count used
- `AdaptiveSparkPlan isFinalPlan=false` → `isFinalPlan=true` marks the transition


In [ ]:
import time

# Set shuffle.partitions=200 to make AQE coalescing visible in the plan.
# With only 4 distinct keys, AQE will coalesce 200 → ≤4 non-empty partitions.
spark.conf.set('spark.sql.shuffle.partitions', '200')

aqe_demo = df.groupBy('key').agg(
    F.sum('id').alias('total_id'),
    F.count('*').alias('cnt')
)

print('── Pre-execution plan (isFinalPlan=false — AQE not yet applied) ──')
aqe_demo.explain(mode='formatted')
# Look for: Exchange hashpartitioning(key#..., 200)
# Spark plans 200 shuffle partitions regardless of how many keys there are.

t0 = time.perf_counter()
aqe_demo.show()
print(f'Took: {time.perf_counter()-t0:.3f}s')

print('\n── Post-execution plan (isFinalPlan=true — AQE decisions locked in) ──')
aqe_demo.explain(mode='formatted')
# Look for:
#   AdaptiveSparkPlan isFinalPlan=true
#   AQEShuffleRead: shows the actual vs. planned partition count after coalescing
# With 4 keys across 200 planned partitions, AQE coalesces to 4 (or fewer) real partitions.

spark.conf.set('spark.sql.shuffle.partitions', '8')
print('\nReset: spark.sql.shuffle.partitions = 8')


### Reading AQE coalescing in the SQL tab

**Pre-execution plan** (`isFinalPlan=false`):
- `Exchange hashpartitioning(key#..., 200)` — Spark planned 200 shuffle partitions

**Post-execution plan** (same query after the action, `isFinalPlan=true`):
- `AQEShuffleRead` node wraps the Exchange — shows the actual coalesced partition count
- With 4 distinct keys across 200 planned partitions, AQE coalesces to ≤ 4 real partitions

**Stages tab:**
- Without AQE: Stage 1 would have 200 tasks (one per planned partition)
- With AQE: Stage 1 runs 4 tasks (one per non-empty coalesced partition)

**The core observation:** the gap between the planned partition count and the actual coalesced count is a direct signal of how well `spark.sql.shuffle.partitions` matches the data. A large gap (200 → 4) means you are over-partitioned for this workload. What that means for tuning — and the two other AQE features that matter for joins and skew — is NB06.


## 8 — Production Debugging Simulation

**Scenario:** "A Spark job aggregating transactions by product category ran in 5 minutes last week. Same code, same cluster. Today it took 20 minutes. Where do you start?"

### Step 1: Jobs tab — isolate the slow job
- Find the job with the highest duration
- **Description column**: tells you which action triggered it (`show`, `count`, `write`, etc.)
- **Stages column**: if the count jumped from 2 to 4, a join strategy likely changed (BHJ → SMJ = 2 extra shuffle stages)

### Step 2: Stages tab — find the bottleneck stage
- Sort by **Duration** — the slowest stage is the target
- Stages with `Exchange` in their description = shuffle-heavy (most likely candidates)
- Check **Input Records** — if one stage processes significantly more records than usual, data volume grew

### Step 3: Task Metrics — is it skew or volume?
Click into the bottleneck stage and look at:
- **Duration histogram**: one bar far to the right = hot key skew (one task >> others)
- **Input Records per task**: one task with 10× the records confirms skew
- *Shuffle Spill (Disk) and GC Time are memory-model signals — covered in NB07.*

### Step 4: SQL tab — read the physical plan for root cause
- `SortMergeJoin` where you expected `BroadcastHashJoin` → table grew past `autoBroadcastJoinThreshold`
- `Filter` node above an `Exchange` → late filter, fix by pushing it upstream
- `AQEShuffleRead coalesced from 200 to 1` → all data in one partition = extreme skew
- *File-level `PushedFilters` (Parquet/Delta/BigQuery predicate pushdown at the storage layer) → NB09.*

### Decision tree
```
Job is slow →
  Jobs tab: Did stage count increase?
    YES → SQL tab: BroadcastHashJoin → SortMergeJoin?
          Fix: F.broadcast() hint or raise autoBroadcastJoinThreshold
    NO  → Stages tab: Which stage is slowest?
           Shuffle stage bottleneck?
             Task Metrics — Duration histogram skewed?
               ONE OUTLIER → Hot key skew
                             Fix: salting (for groupBy) or AQE skew join hint (NB06)
               ALL SLOW    → Data volume grew
                             Check Input Records vs. baseline
           Non-shuffle stage bottleneck?
             SQL tab — Filter above Exchange?
             Fix: move filter before the groupBy or join
```

Run the cells below to see a "before" and "regressed" plan side by side — the `explain()` output surfaces the root cause in seconds.


In [ ]:
import time

# ── Normal version: early filter + broadcast join ────────────────────────────
# This is the "last week" state — fast, minimal shuffles.
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))

normal_job = (
    txn_df
    .filter(F.col('amount') > 30.0)                     # early filter: pushed before the join
    .join(F.broadcast(products_df), 'product_id')        # BroadcastHashJoin: no shuffle on small side
    .groupBy('category')
    .agg(F.sum('amount').alias('revenue'), F.count('*').alias('txn_count'))
)
t0 = time.perf_counter()
normal_job.show()
print(f'Normal (BHJ + early filter): {time.perf_counter()-t0:.3f}s')
print()
print('Normal physical plan:')
normal_job.explain(mode='formatted')

print('\n' + '='*70)
print('SIMULATED REGRESSION: broadcast disabled + late filter')
print('='*70 + '\n')

# ── Regressed version: late filter + sort-merge join ─────────────────────────
# Simulates: the products table grew past broadcast threshold, and a refactor moved
# the filter downstream of the join.
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')  # disable broadcast

regressed_job = (
    txn_df
    .join(products_df, 'product_id')                     # SortMergeJoin: both sides shuffled
    .filter(F.col('amount') > 30.0)                      # late filter: runs AFTER the join shuffle
    .groupBy('category')
    .agg(F.sum('amount').alias('revenue'), F.count('*').alias('txn_count'))
)
t0 = time.perf_counter()
regressed_job.show()
print(f'Regressed (SMJ + late filter): {time.perf_counter()-t0:.3f}s')
print()
print('Regressed physical plan:')
regressed_job.explain(mode='formatted')

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', str(10 * 1024 * 1024))
print('\n--- Findings ---')
print('Normal:    BroadcastHashJoin + Filter below Exchange (1 shuffle)')
print('Regressed: SortMergeJoin + Filter above Exchange    (2 shuffles + late filter)')
print('Fix:       use F.broadcast(products_df) hint; move filter before join')
